In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# ==============================
# Load data
# ==============================
file_path = "Figure_S7G_Foci_Number_WNK463.csv"
df = pd.read_csv(file_path)

# Melt to long format
df_long = df.melt(var_name='Condition', value_name='Foci Count')

# Parse drug and osmolarity info
df_long['Drug'] = df_long['Condition'].apply(
    lambda x: 'DMSO' if 'DMSO' in x else 'WNK463' if 'WNK463' in x else 'Unknown'
)
df_long['Osmolarity'] = df_long['Condition'].apply(
    lambda x: '100_mOsm_per_L' if '100' in x else
              '300_mOsm_per_L' if '300' in x else
              '700_mOsm_per_L' if '700' in x else 'Unknown'
)

# Define order
osm_order = ['300_mOsm_per_L', '100_mOsm_per_L', '700_mOsm_per_L']
drug_order = ['DMSO', 'WNK463']
condition_order = [
    f"{drug}_{osm}" for drug in drug_order for osm in osm_order
    if f"{drug}_{osm}" in df.columns
]
df_long['Condition'] = pd.Categorical(df_long['Condition'], categories=condition_order, ordered=True)

# ==============================
# Plot settings
# ==============================
plt.rcParams['font.family'] = 'Arial'
fig, ax = plt.subplots(figsize=(4, 3.5))

# Color palette for osmolarity
osm_colors = {
    '300_mOsm_per_L': '#e8e8e8',
    '100_mOsm_per_L': '#ffe5b6',
    '700_mOsm_per_L': '#49c1bb'
}

# Violin plot (half)
vp = sns.violinplot(
    x='Condition', y='Foci Count', data=df_long,
    hue='Osmolarity', order=condition_order,
    inner=None, cut=0, scale='width', palette=osm_colors,
    linewidth=0, dodge=False, ax=ax
)

# Clip to left half
for violin in ax.collections:
    path = violin.get_paths()[0]
    vertices = path.vertices
    center_x = np.median(vertices[:, 0])
    vertices[:, 0] = np.minimum(vertices[:, 0], center_x)
    path.vertices = vertices

# ==============================
# Boxplot overlay (white fill)
# ==============================
sns.boxplot(
    x='Condition', y='Foci Count', data=df_long,
    showcaps=False, showfliers=False, width=0.35,
    boxprops={'facecolor': 'white', 'edgecolor': 'black', 'zorder': 4},
    medianprops={'color': 'black', 'linewidth': 1.2},
    whiskerprops={'color': 'black'}, capprops={'color': 'black'},
    dodge=False, ax=ax
)

# ==============================
# Stripplot (white dots, black edge)
# ==============================
sns.stripplot(
    x='Condition', y='Foci Count', data=df_long,
    jitter=True, size=3.5, alpha=1,
    color='white', edgecolor='black', linewidth=0.6,
    dodge=False, ax=ax
)

# ==============================
# Mean lines
# ==============================
group_means = df_long.groupby('Condition')['Foci Count'].mean()
for i, cond in enumerate(condition_order):
    ax.hlines(
        y=group_means[cond], xmin=i - 0.2, xmax=i + 0.2,
        colors='black', linewidth=1.2, zorder=6
    )

# Remove legend
if ax.get_legend():
    ax.get_legend().remove()

# Add vertical separator between DMSO and WNK463 groups
ax.axvline(x=2.5, color='black', linestyle='--', linewidth=1)

# Axis labels
ax.set_xlabel("")
ax.set_ylabel("# of Foci / cell", fontsize=11)
ax.set_xticklabels(condition_order, rotation=45, ha='right', fontsize=9)

# Clean look
sns.despine(trim=False, top=False, right=False)
plt.tight_layout()

# ==============================
# Save
# ==============================
plt.savefig(
    "Figure_S7G_HalfViolin_Box_MeanLine_Grouped.pdf",
    format="pdf", dpi=300, bbox_inches="tight", transparent=True
)
plt.show()


In [ ]:
import pandas as pd
import scipy.stats as stats
from itertools import combinations
from statsmodels.stats.multitest import multipletests

# --- Load data ---
file_path = "Figure_S7G_Foci_Number_WNK463.csv"
df = pd.read_csv(file_path)

# --- Reshape to long format ---
df_long = df.melt(var_name="Condition", value_name="Foci_Count").dropna()
df_long["Drug"] = df_long["Condition"].apply(lambda x: "DMSO" if "DMSO" in x else "WNK463")
df_long["Osmolarity"] = df_long["Condition"].apply(lambda x: "300" if "300" in x else ("100" if "100" in x else "700"))
df_long["Group"] = df_long["Drug"] + "_" + df_long["Osmolarity"]

# --- Prepare groups ---
groups = df_long.groupby("Group")["Foci_Count"].apply(list)
pairs = list(combinations(groups.index, 2))

# --- Mann–Whitney U tests ---
results = []
for g1, g2 in pairs:
    u_stat, p_val = stats.mannwhitneyu(groups[g1], groups[g2], alternative='two-sided')
    results.append({"Group1": g1, "Group2": g2, "U_stat": u_stat, "p_uncorrected": p_val})

results_df = pd.DataFrame(results)

# --- FDR correction (Benjamini-Hochberg) ---
reject, p_corrected, _, _ = multipletests(results_df["p_uncorrected"], method='fdr_bh')
results_df["p_corrected_FDR"] = p_corrected
results_df["Significant"] = reject

# --- Sort by corrected p-value ---
results_df = results_df.sort_values("p_corrected_FDR")

# --- Save or display ---
print(results_df)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# ==============================
# Load data
# ==============================
file_path = "Figure_S7H_Foci_Size_WNK463.csv"
df = pd.read_csv(file_path)

# Melt to long format
df_long = df.melt(var_name='Condition', value_name='Foci Count')

# Parse drug and osmolarity info
df_long['Drug'] = df_long['Condition'].apply(
    lambda x: 'DMSO' if 'DMSO' in x else 'WNK463' if 'WNK463' in x else 'Unknown'
)
df_long['Osmolarity'] = df_long['Condition'].apply(
    lambda x: '100_mOsm_per_L' if '100' in x else
              '300_mOsm_per_L' if '300' in x else
              '700_mOsm_per_L' if '700' in x else 'Unknown'
)

# Define order
osm_order = ['300_mOsm_per_L', '100_mOsm_per_L', '700_mOsm_per_L']
drug_order = ['DMSO', 'WNK463']
condition_order = [
    f"{drug}_{osm}" for drug in drug_order for osm in osm_order
    if f"{drug}_{osm}" in df.columns
]
df_long['Condition'] = pd.Categorical(df_long['Condition'], categories=condition_order, ordered=True)

# ==============================
# Plot settings
# ==============================
plt.rcParams['font.family'] = 'Arial'
fig, ax = plt.subplots(figsize=(4, 3.5))

# Color palette for osmolarity
osm_colors = {
    '300_mOsm_per_L': '#e8e8e8',
    '100_mOsm_per_L': '#ffe5b6',
    '700_mOsm_per_L': '#49c1bb'
}

# Violin plot (half)
vp = sns.violinplot(
    x='Condition', y='Foci Count', data=df_long,
    hue='Osmolarity', order=condition_order,
    inner=None, cut=0, scale='width', palette=osm_colors,
    linewidth=0, dodge=False, ax=ax
)

# Clip to left half
for violin in ax.collections:
    path = violin.get_paths()[0]
    vertices = path.vertices
    center_x = np.median(vertices[:, 0])
    vertices[:, 0] = np.minimum(vertices[:, 0], center_x)
    path.vertices = vertices

# ==============================
# Boxplot overlay (white fill)
# ==============================
sns.boxplot(
    x='Condition', y='Foci Count', data=df_long,
    showcaps=False, showfliers=False, width=0.35,
    boxprops={'facecolor': 'white', 'edgecolor': 'black', 'zorder': 4},
    medianprops={'color': 'black', 'linewidth': 1.2},
    whiskerprops={'color': 'black'}, capprops={'color': 'black'},
    dodge=False, ax=ax
)

# ==============================
# Stripplot (white dots, black edge)
# ==============================
sns.stripplot(
    x='Condition', y='Foci Count', data=df_long,
    jitter=True, size=1, alpha=1,
    color='white', edgecolor='black', linewidth=0.1,
    dodge=False, ax=ax
)

# ==============================
# Mean lines
# ==============================
group_means = df_long.groupby('Condition')['Foci Count'].mean()
for i, cond in enumerate(condition_order):
    ax.hlines(
        y=group_means[cond], xmin=i - 0.2, xmax=i + 0.2,
        colors='black', linewidth=1.2, zorder=6
    )

# Remove legend
if ax.get_legend():
    ax.get_legend().remove()

# Add vertical separator between DMSO and WNK463 groups
ax.axvline(x=2.5, color='black', linestyle='--', linewidth=1)

# Axis labels
ax.set_xlabel("")
ax.set_ylabel("Foci Size (nm)", fontsize=11)
ax.set_xticklabels(condition_order, rotation=45, ha='right', fontsize=9)

# Clean look
sns.despine(trim=False, top=False, right=False)
plt.tight_layout()

# ==============================
# Save
# ==============================
plt.savefig(
    "Figure_S7H_HalfViolin_Box_MeanLine_Grouped.pdf",
    format="pdf", dpi=300, bbox_inches="tight", transparent=True
)
plt.show()


In [ ]:
import pandas as pd
import scipy.stats as stats
from itertools import combinations
from statsmodels.stats.multitest import multipletests

# --- Load data ---
file_path = "Figure_S7H_Foci_size_WNK463.csv"
df = pd.read_csv(file_path)

# --- Reshape to long format ---
df_long = df.melt(var_name="Condition", value_name="Foci_Count").dropna()
df_long["Drug"] = df_long["Condition"].apply(lambda x: "DMSO" if "DMSO" in x else "WNK463")
df_long["Osmolarity"] = df_long["Condition"].apply(lambda x: "300" if "300" in x else ("100" if "100" in x else "700"))
df_long["Group"] = df_long["Drug"] + "_" + df_long["Osmolarity"]

# --- Prepare groups ---
groups = df_long.groupby("Group")["Foci_Count"].apply(list)
pairs = list(combinations(groups.index, 2))

# --- Mann–Whitney U tests ---
results = []
for g1, g2 in pairs:
    u_stat, p_val = stats.mannwhitneyu(groups[g1], groups[g2], alternative='two-sided')
    results.append({"Group1": g1, "Group2": g2, "U_stat": u_stat, "p_uncorrected": p_val})

results_df = pd.DataFrame(results)

# --- FDR correction (Benjamini-Hochberg) ---
reject, p_corrected, _, _ = multipletests(results_df["p_uncorrected"], method='fdr_bh')
results_df["p_corrected_FDR"] = p_corrected
results_df["Significant"] = reject

# --- Sort by corrected p-value ---
results_df = results_df.sort_values("p_corrected_FDR")

# --- Save or display ---
print(results_df)